# Irreversibility project: run everything from Jupyter on the DGX

Every task runs **inside this Jupyter session**, on the GPU node that `srun` gave you.
Nothing runs on the login node. Kernel: the default **Python 3** is fine; each task activates the
project environment by itself.

**Which session to open (in tmux on the login node, then start Jupyter in it):**

| Session | Use it for | srun resources |
|---|---|---|
| S1 setup + data | Phases 0, A, B | `--gres=gpu:a100_1g.5gb:1 -c 16 --mem=64G --time=04:00:00` |
| S2 model work | Phases C, D, E, F | `--gres=gpu:a100:1 -c 16 --mem=96G --time=12:00:00` |

The 5 GB / 10 GB GPU slices are **too small** for a 7B model. For S2, use `gpu:a100:1`, or else
`gpu:a100_3g.20gb:1` (slower; the agent cells then use `batch=4`).
If the session's time runs out mid-task, just open a new session and run the same cell again:
every task resumes where it stopped.

In [ ]:
import os, subprocess, getpass, time
HOME = os.path.expanduser("~")
PROJ = f"{HOME}/irrev-project"
ENV  = f"{HOME}/envs/irrev"
os.chdir(PROJ)

def sh(cmd, env=None):
    """Run a shell command and stream its output into the notebook. Raises an error if it fails."""
    t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, **(env or {})}, executable="/bin/bash")
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    print(f"\n--- exit code {p.returncode}, {(time.time()-t0)/60:.1f} min")
    if p.returncode != 0:
        raise RuntimeError(f"command failed: {cmd}")

def run(stage, model="qwen7b", *extra):
    """Run one pipeline task on this node (same as the runbook's job.sh commands)."""
    ncpu = os.environ.get("SLURM_CPUS_ON_NODE", "4")
    sh(f"bash scripts/cluster/job.sh {stage} {model} {' '.join(map(str, extra))}",
       env={"SLURM_CPUS_PER_TASK": ncpu})

sh("hostname; echo CPUs: $SLURM_CPUS_ON_NODE; nvidia-smi --query-gpu=name,memory.total --format=csv; git log --oneline -1")

## Phase 0: does this compute node have internet? (decides where setup can run)

In [ ]:
sh("for u in https://huggingface.co https://pypi.org/simple/ https://download.pytorch.org; do printf '%-35s ' $u; curl -s -o /dev/null -m 10 -w '%{http_code}\\n' $u || echo NO; done")

All three print `200` → do Phase A here.
If they print `NO` / `000` → this node is offline. Then A1 and A3 must run once on the login node
(`bash scripts/cluster/02_setup_env.sh` and the `03_download.sh` lines). They only install packages
and download files, so they are light. Tell Claude if that is not allowed on your cluster.

## Phase A: setup (once for the whole team, ~1 h)

In [ ]:
sh("bash scripts/cluster/02_setup_env.sh")      # A1: environment, ~20-30 min. Ends with 'scienceworld import OK'

In [ ]:
# A2: Hugging Face login. The token is typed into a hidden box: it is NOT saved in this notebook.
tok = getpass.getpass("Hugging Face token: ")
sh(f'{ENV}/bin/python -c "import os; from huggingface_hub import login; login(token=os.environ[\'HF_TOKEN_TMP\'])"',
   env={"HF_TOKEN_TMP": tok, "HF_HOME": f"{HOME}/hf_cache"})
del tok

In [ ]:
# A3: downloads, ~10-30 min in total
sh("bash scripts/cluster/03_download.sh qwen7b")
sh("bash scripts/cluster/03_download.sh llama8b")
sh("bash scripts/cluster/03_download.sh bytesized")

## Phase B: data (session S1)

In [ ]:
run("tests")            # ⛔ all PASS and 'Demo finished' (~5 min)

In [ ]:
run("data")             # expect 1953 records, 858/305/790, 60.5%, 130 tasks (65 pairs)

In [ ]:
run("sciworld_pilot")   # a few records, no error (~5 min)

In [ ]:
run("sciworld")         # 30-90 min with 16 CPUs. Re-run this cell if the session ends: it resumes

In [ ]:
run("sciworld_check")   # ⛔ read the samples; send this output to Claude

In [ ]:
run("build")            # ⛔ >= 2000 records, minimal-pair share >= 30%

Then commit the frozen dataset from a terminal on the login node (git is light):
`cd ~/irrev-project && git add data/ requirements.lock.txt && git commit -m "freeze dataset v1" && git push`

## Phase C: Qwen (session S2, full A100)

In [ ]:
MODEL = "qwen7b"         # for Phase D change to "llama8b" and run C1-C9 again

In [ ]:
run("smoke", MODEL)      # C1 ⛔ must end with ALL CHECKS PASSED. Send the output to Claude.

In [ ]:
run("hidden", MODEL)     # C2 ~10-20 min

In [ ]:
run("probes", MODEL)     # C3 30-90 min (CPU-heavy, uses all CPUs of the session)

In [ ]:
run("stated", MODEL)     # C4 ~10-20 min

In [ ]:
run("rq2", MODEL)        # C5 ~2 min

⛔ Commit `docs/preregistration.md` **before** the agent runs.

In [ ]:
run("agent", MODEL)          # C6 20-50 min (on a 20 GB slice the batch is too big: tell Claude)

In [ ]:
run("agent_careful", MODEL)  # C7 20-50 min

In [ ]:
run("knowledge", MODEL)      # C8 ~5 min

In [ ]:
run("analysis", MODEL)       # C9 ~1 min: gap = K - A

## Phase E: bonus, steering + RQ5 (Qwen only)
The 28 steering runs take 10-18 h in total. Run them across several sessions with IDs 0-27; each ID
is independent, so teammates can split them (e.g. 0-6, 7-13, 14-20, 21-27).

In [ ]:
run("steerdir", "qwen7b")

In [ ]:
for i in range(0, 7):        # change the range per person / per session
    run("steer", "qwen7b", i)

In [ ]:
run("steer_analysis", "qwen7b")   # after all 28 are done

In [ ]:
run("rq5", "qwen7b")              # 30-60 min

## Phase F: figures (after both models)

In [ ]:
run("figures"); sh("ls results/figures")